In [1]:
from common_utils.utils import chunks
from tqdm import tqdm
from matplotlib import pyplot as plt
from pylab import rcParams
import seaborn as sns

from IPython.display import display, HTML
import pandas as pd

import logging
from datetime import datetime, timedelta
import pandas as pd

from common_utils.notebook_utils import sign_notebook

pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 200)
from pandas.io.json import json_normalize

rcParams['figure.figsize'] = 15, 8

sns.set(rc={'figure.figsize':(12,6)})

sns.set(style="whitegrid", color_codes=True)
sns.despine()


from dbutils.query_utils import get_interval_clauses, run_select
from dbutils.utils import get_nabu_payments_client, get_nabu_ledger_client
from dbutils.query_utils import get_select_query, run_select, get_interval_clauses
from customers_analytics.fraud_analysis.fraud_db_utils import get_fraud_client
from customers_analytics.nabu.nabu_db_utils import (
    get_nabu_client_internal, 
    get_nabu_backoffice_client, 
    get_nabu_client, 
    get_nabu_client_private, 
    get_nabu_backoffice_client_prod, 
    categorize_blockchain_error_reasons
)

INFO:root:Role:data-science Environment:internal


<Figure size 864x432 with 0 Axes>

In [3]:
# from customers_analytics.nabu.internal_tables.payments import update_payments_account

client = get_nabu_payments_client()
client_internal = get_nabu_client_internal()

## Fetch data from card_payments_fraud_stats

In [5]:
# query=\
# '''
# select
#      id
#     ,user_id
#     ,payment_reference
#     ,case when lower(card_type) like '%visa%' then 'visa'
#           when lower(card_type) like '%master%' then 'mastercard'
#           else 'other'
#      end as card_type
# from card_payments_fraud_stats
# '''

In [7]:
# df_card_pay = client_internal.dataframe_from_query(query)

## Merge `payments.payments_events` with `payments.transaction`

### Trying to use `payment_reference` in merge condition if `payment_id` is Null

In [64]:
query=\
'''
select
  pay_events.id as pay_events_id
 ,pay_events.user_id
 ,trans.beneficiary as id
 ,case when pay_events.event_type = 'FRAUD' then 1 else 0 end as is_fraud
 ,pay_events.payment_id
 ,pay_events.payment_reference
 ,trans_view.extra_attributes ->> 'processingAccount' as processing_account
from payments.payments_events as pay_events
left join payments.transaction as trans 
     on (pay_events.payment_id = trans.id) or (pay_events.payment_reference = trans.extra_attributes -> 'cardProvider' ->> 'payment_reference')
left join payments.transaction_view as trans_view 
     on ((pay_events.payment_id = trans_view.id) and (pay_events.user_id = trans_view.account_user)) or
        (pay_events.payment_reference = trans_view.extra_attributes -> 'cardProvider' ->> 'payment_reference')
--where payment_id is not null
'''

In [ ]:
# df_pay_events = client.dataframe_from_query(query)

## Skip use of `payment_reference` in merge condition: merge only by transaction's `id`

### Mapping between `transaction`, `transaction_view` and `payments_events`

In [ ]:
#payments.transaction.id == payments.transaction_view.id == payments.payments_events.payment_id

# select * from payments.transaction
# where payments.transaction.id = '6cabc0b2-b15b-4e45-919e-aeaab96bffd2'
# --user_account = ab64786e-b3e6-4db3-9e54-7ffdd2c8dfd2

# select * from payments.transaction_view
# where payments.transaction_view.id = '6cabc0b2-b15b-4e45-919e-aeaab96bffd2'
# --account_user = 17526e1c-79f4-4c3b-be05-64fd128defb2
# --beneficiary_user = 17526e1c-79f4-4c3b-be05-64fd128defb2
# --user_account = ab64786e-b3e6-4db3-9e54-7ffdd2c8dfd2

# select * from payments.payments_events 
# where payment_id = '6cabc0b2-b15b-4e45-919e-aeaab96bffd2'
# --user_id = 17526e1c-79f4-4c3b-be05-64fd128defb2

## Running Query without usage of explisit json serialization

In [44]:
from customers_analytics.payments.enums import PaymentLegType

### Perform Monthly Calc

In [67]:
start_date = "2022-05-01T00:00+0000"
end_date   = "2022-06-01T00:00+0000"

In [68]:
%%time
transactions = run_select(
    client=client,
    table="payments.transaction left join payments.transaction_state "
    "on payments.transaction.id=payments.transaction_state.transaction "
    "left join payments.transaction_leg on payments.transaction_leg.id=payments.transaction.id",
    cols=[
        "payments.transaction.id",
        "payments.transaction.created_at",
        "payments.transaction_state.inserted_at", 
        "payments.transaction.beneficiary",
        "payments.transaction_leg.usd_amount as usd_amount",
        "payments.transaction.extra_attributes ->> 'recharge_state' is not null as is_recharge",
        "payments.transaction.extra_attributes -> 'cardProvider' ->> 'payment_reference' as payment_reference",
        # "payments.transaction_view.extra_attributes as transaction_view_extra_attributes"
    ],
    between_clauses={"payments.transaction_state.inserted_at": (start_date, end_date)},
    simple_clauses={
        "payments.transaction_state.state": ("COMPLETE", True),
        "payments.transaction.type": ("REFUND", False),
        "payments.transaction_leg.type": (PaymentLegType.PRINCIPAL, True),
    }
)

ERROR:root:server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.

ERROR:root:Error: server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.
. Retrying in 3 seconds


CPU times: user 1.3 s, sys: 184 ms, total: 1.48 s
Wall time: 10min 58s


## SQL query with json str serialization

### Using multiple json Serialization leads to overloaded query state and abnormally increases the total calculation time 

In [ ]:
# %%time
# transactions = run_select(
#     client=client,
#     table="payments.transaction left join payments.transaction_state "
#     "on payments.transaction.id=payments.transaction_state.transaction "
#     "left join payments.transaction_leg on payments.transaction_leg.id=payments.transaction.id "
#     "left join payments.transaction_view on payments.transaction_view.id=payments.transaction.id",
#     cols=[
#         "payments.transaction.id",
#         # "payments.account.user as user_id",
#         # "payments.account.partner as partner",
#         # "payments.account.extra_attributes -> 'card' ->> 'type' as card_type",
#         # "payments.account.extra_attributes -> 'card' ->> 'issuer_country' as card_issuer_country",
#         # "payments.account.extra_attributes -> 'billing_address' ->> 'country' as billing_country", 
#         "payments.transaction.created_at",
#         "payments.transaction_state.inserted_at",
#         "payments.transaction.beneficiary",        
#         "payments.transaction_leg.usd_amount as usd_amount",
#         # "payments.transaction_state.state",
#         "payments.transaction.extra_attributes ->> 'recharge_state' is not null as is_recharge",
#         # "payments.transaction.extra_attributes -> 'cardProvider' ->> 'payment_reference' as payment_reference",
#         "payments.transaction_view.extra_attributes ->> 'processingAccount' as processing_account"
#     ],
#     between_clauses={"payments.transaction_state.inserted_at": (start_date, end_date)},
#     simple_clauses={
#         # "payments.account.account_ref": ("card", True),
#         "payments.transaction_state.state": ("COMPLETE", True),
#         "payments.transaction.type": ("REFUND", False),
#         "payments.transaction_leg.type": (PaymentLegType.PRINCIPAL, True),
#     },
#     # in_clauses={"payments.account.partner": (["CARDPROVIDER", "EVERYPAY"], True)},
# )

ERROR:root:canceling statement due to conflict with recovery
DETAIL:  User query might have needed to see row versions that must be removed.

ERROR:root:Error: canceling statement due to conflict with recovery
DETAIL:  User query might have needed to see row versions that must be removed.
. Retrying in 3 seconds
ERROR:root:canceling statement due to conflict with recovery
DETAIL:  User query might have needed to see row versions that must be removed.

ERROR:root:Error: canceling statement due to conflict with recovery
DETAIL:  User query might have needed to see row versions that must be removed.
. Retrying in 3 seconds


In [76]:
transactions = transactions.drop_duplicates("id")

## Join with TransactionView

In [78]:
query=\
'''
select
     distinct
     payments.transaction_view.id
    ,payments.transaction_view.extra_attributes ->> 'processingAccount' as processing_account
from payments.transaction_view
where inserted_at between '{}' and '{}'
'''.format(start_date, end_date)

df_transaction_view = client.dataframe_from_query(query)

ERROR:root:server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.

ERROR:root:Error: server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.
. Retrying in 3 seconds
ERROR:root:canceling statement due to conflict with recovery
DETAIL:  User query might have needed to see row versions that must be removed.

ERROR:root:Error: canceling statement due to conflict with recovery
DETAIL:  User query might have needed to see row versions that must be removed.
. Retrying in 3 seconds


In [79]:
trnx_joined_df = transactions.merge(df_transaction_view, 
                                     on=['id'],
                                     how='inner',
                                     validate='1:1')

In [80]:
trnx_joined_df = trnx_joined_df.drop_duplicates("id")

In [13]:
trnx_joined_df.shape

(269695, 8)

## Get only `checkoutdotcom` processing records

In [81]:
trnx_joined_df[~trnx_joined_df.processing_account.isnull()].head()

,id,created_at,inserted_at,beneficiary,usd_amount,is_recharge,payment_reference,processing_account
1,b937092b-ae41-4307-9941-9d0258ffa8e2,2022-05-01 03:37:37.095,2022-05-01 03:38:12.436956,1fe70e39-06e0-4f47-b1a8-2d28d86fa17f,240.000000000,False,pay_inbqco2nicwercewugkm3oio7i,checkoutdotcom_uk1
2,561d4997-0460-4b25-bfb1-a498856d6921,2022-05-01 03:38:36.766,2022-05-01 03:39:26.194660,25bce4b1-387c-42bd-9807-6d7fa1770aa4,15.795000000,False,pay_vier65zxtpfefbymnxdq6nznzy,checkoutdotcom_uk1
5,674aecae-6db7-4c04-b505-2f29c727317d,2022-05-01 03:45:49.280,2022-05-01 03:46:00.353727,dad3b904-fca9-4ca2-80a9-e9407b2a102b,100.000000000,False,pay_togxzyyboiqetlwnuyedvpufpu,checkoutdotcom_us1
6,3cc3a9c1-47c4-4d81-bd9a-8f35826b984a,2022-05-01 03:37:45.001,2022-05-01 03:38:35.758218,33bbd20d-4527-4e01-9d1c-430209ecd975,25.000000000,False,None,USD3D1
7,fa34d8e8-ed0d-4c52-a5b2-f2d047012430,2022-05-01 05:51:33.948,2022-05-01 05:51:42.796413,46ab0965-4efd-4ef0-b86c-244ada9b0464,60.000000000,False,pi_3KuVgwKDv3vdWtQB1cvIjaWj,stripe_us1


In [82]:
trnx_joined_df.processing_account = trnx_joined_df.processing_account.fillna('')

In [83]:
trnx_joined_df = trnx_joined_df[trnx_joined_df.processing_account.str.contains('checkoutdotcom')]

## Fetch fraud from `payments.account`

In [84]:
%%time
payments = run_select(
    client=client,
    table="payments.account",
    cols=[
        "payments.account.id as pay_id",
        "payments.account.user as user_id",
        "payments.account.partner as partner",
        "case when lower(payments.account.extra_attributes -> 'card' ->> 'type') like '%visa%' then 'visa' \
              when lower(payments.account.extra_attributes -> 'card' ->> 'type') like '%master%' then 'mastercard' \
              else 'other' \
         end as card_type",       
        "payments.account.extra_attributes -> 'card' ->> 'issuer_country' as card_issuer_country",
        "payments.account.extra_attributes -> 'billing_address' ->> 'country' as billing_country"
    ],
    between_clauses={"payments.account.inserted_at": (start_date, end_date)},
    simple_clauses={"payments.account.account_ref": ("card", True)},
    in_clauses={"payments.account.partner": (["CARDPROVIDER", "EVERYPAY"], True)}
)

CPU times: user 324 ms, sys: 52 ms, total: 376 ms
Wall time: 5min 1s


In [85]:
payments = payments[payments.card_type!='other']

In [15]:
payments.head()

,pay_id,user_id,partner,card_type,card_issuer_country,billing_country
0,18d5b92a-ac80-4d1a-8abb-5fd554ab0959,e7b89ddf-aae6-48f1-982f-312d4813324c,CARDPROVIDER,visa,BR,BR
4,22cdbcf0-4374-46b8-80ad-cbc421e2a716,34ce9e71-a33c-44f8-8665-448dff81ac2b,CARDPROVIDER,visa,US,US
5,cd18f06e-8ac0-4cd8-aa7c-8cbd220ddb51,2aa28628-714b-4c5c-acff-7ca72dd1d681,CARDPROVIDER,visa,US,US
7,aa0898e4-e683-4beb-80da-5819ef4c0832,e4bee2b4-03f1-4975-8919-acc899ba41ef,CARDPROVIDER,mastercard,US,US
9,ce4755c8-a377-4235-93f6-d937a7c33a06,ce046d91-5392-4582-ab1b-ff3369d4918f,CARDPROVIDER,visa,US,US


## Merging by `payments_account.id == payments.transaction.beneficiary`

In [86]:
trnx_pay_df = trnx_joined_df.merge(payments, 
                                     left_on=['beneficiary'],
                                     right_on=['pay_id'],
                                     how='inner',
                                     validate='m:1')

In [98]:
trnx_pay_df.head()

,id,created_at,inserted_at,beneficiary,usd_amount,is_recharge,payment_reference,processing_account,pay_id,user_id,partner,card_type,card_issuer_country,billing_country
0,674aecae-6db7-4c04-b505-2f29c727317d,2022-05-01 03:45:49.280,2022-05-01 03:46:00.353727,dad3b904-fca9-4ca2-80a9-e9407b2a102b,100.000000000,False,pay_togxzyyboiqetlwnuyedvpufpu,checkoutdotcom_us1,dad3b904-fca9-4ca2-80a9-e9407b2a102b,fec9d322-95cf-4c17-a7d7-06cf2296a8e2,CARDPROVIDER,visa,US,GT
1,837de733-be4e-40cc-8354-e9c087654a61,2022-05-01 03:48:40.455,2022-05-01 03:48:49.590958,010bf6b7-244f-4969-a121-1570deaf3f5a,100.000000000,False,pay_kiuachb5htjevfy4jn4buoceti,checkoutdotcom_uk1,010bf6b7-244f-4969-a121-1570deaf3f5a,21633d2a-557a-477c-964c-610b682e14be,CARDPROVIDER,mastercard,CA,CA
2,fe489b38-2f94-4415-970b-a163b5fe73d2,2022-05-01 05:18:16.923,2022-05-01 05:18:26.547264,71739873-4229-483f-9183-cd6ed60b58d2,200.000000000,False,pay_itma7ecqniyuxk7mhhf4w7jwwa,checkoutdotcom_us1,71739873-4229-483f-9183-cd6ed60b58d2,cc6f217b-7189-45a4-b73e-863e3014c221,CARDPROVIDER,visa,CA,CA
3,7fa27b5c-7b94-47a5-8bc1-30aae65e3c4b,2022-05-01 05:18:28.785,2022-05-01 05:18:39.243731,b850ffab-2cf8-43a9-95f6-854f638c10c7,5.000000000,False,pay_xef2hexuqm5exmp4diuo4bbedi,checkoutdotcom_uk1,b850ffab-2cf8-43a9-95f6-854f638c10c7,741dd326-4677-4ae1-82dd-11c65f9e67fb,CARDPROVIDER,visa,AU,AU
4,ebec7f0c-77a4-48e2-99c8-cd15003f4046,2022-05-01 05:17:12.981,2022-05-01 05:17:26.215449,b850ffab-2cf8-43a9-95f6-854f638c10c7,20.000000000,False,pay_lq7inap3lf4uhghqofkilpfa44,checkoutdotcom_uk1,b850ffab-2cf8-43a9-95f6-854f638c10c7,741dd326-4677-4ae1-82dd-11c65f9e67fb,CARDPROVIDER,visa,AU,AU


## Fetch fraud from `payments_events`

In [87]:
payments_events = \
run_select(
    client,
    table='"payments"."payments_events"',
    cols=["event_name", 
          "event_type",
          "case when event_type = 'FRAUD'   then 1 else 0 end as is_fraud",
          "case when event_type = 'DISPUTE' then 1 else 0 end as is_dispute",
          "payment_reference", 
          "updated_at as fraud_reported_date"],
    between_clauses={"updated_at": (start_date, end_date)},
    #simple_clauses={"event_type": ("FRAUD", True)},
)

In [97]:
payments_events.head()

,event_name,event_type,is_fraud,is_dispute,payment_reference,fraud_reported_date
0,10.4,FRAUD,1,0,pay_phd4hwcybque7or2lwglu7akyq,2022-05-02 07:47:12.593
1,LOST,DISPUTE,0,1,pay_sic2zwh5i54uvgiaec3s2rhx3y,2022-05-02 07:47:13.034
2,10.4,FRAUD,1,0,pay_sic2zwh5i54uvgiaec3s2rhx3y,2022-05-02 07:47:13.040
3,LOST,DISPUTE,0,1,pay_ocacwupjd6ne7ifxtwhu4dbpuy,2022-05-02 07:47:13.499
4,10.4,FRAUD,1,0,pay_ocacwupjd6ne7ifxtwhu4dbpuy,2022-05-02 07:47:13.504


In [27]:
payments_events.event_type.unique()

array(['FRAUD', 'DISPUTE', 'UPDATE', 'OTHER', 'ABANDONED', 'REFUND',
       'CHARGEBACK'], dtype=object)

## Merging by `payments_events.payment_reference == payments.transaction.payment_reference`

In [88]:
trnx_pay_all_df = trnx_pay_df.merge(payments_events, 
                                     on=['payment_reference'],
                                     how='inner',
                                     validate='1:m')

In [89]:
trnx_pay_all_df['fraud_reported_date_day'] = trnx_pay_all_df['fraud_reported_date'].apply(lambda x: datetime.strftime(x, '%Y-%m-%d'))

## Fetch data from internal `payments_account`

In [38]:
# query=\
# '''
# with pay_acc as(
# select 
#      id,
#      type,
#      partner,
#      product,
#      account_ref,
#      currency,
#      agent_ref,
#      user,
#      inserted_at,
#      updated_at,
#      last_state,
#      ach_bank_account_type,
#      ach_bank_name,
#      billing_city,
#      billing_country,
#      card_acquirer_name,
#      card_bin,
#      card_funding_source,
#      card_issuer,
#      card_issuer_country,
#      case when lower(card_type) like '%visa%' then 'visa'
#           when lower(card_type) like '%master%' then 'mastercard'
#           else 'other'
#      end as card_type
#      ,user_id 
# from payments_account
# )
# select * from pay_acc
# where 1=1
# --and billing_country in ('GB', 'US') 
# and card_type in ('visa', 'mastercard')
# --and card_acquirer_name = 'CHECKOUTDOTCOM'
# '''

# df_payments_account = client_internal.dataframe_from_query(query)

## Check fraud information for `mastercard` and `visa`

In [33]:
trnx_pay_all_df[(trnx_pay_all_df.card_type=='mastercard')&(trnx_pay_all_df.is_fraud== 1.)].head()

,id,created_at,inserted_at,beneficiary,usd_amount,is_recharge,payment_reference,processing_account,pay_id,user_id,partner,card_type,card_issuer_country,billing_country,event_name,event_type,is_fraud,is_dispute,fraud_reported_date,fraud_reported_date_day


In [99]:
trnx_pay_all_df[(trnx_pay_all_df.card_type=='visa')&(trnx_pay_all_df.is_fraud.isin([0.,1.]))].head()

,id,created_at,inserted_at,beneficiary,usd_amount,is_recharge,payment_reference,processing_account,pay_id,user_id,partner,card_type,card_issuer_country,billing_country,event_name,event_type,is_fraud,is_dispute,fraud_reported_date,fraud_reported_date_day
0,ecba3042-0c98-4765-b98d-6b71ca491b9c,2022-05-02 21:46:38.144,2022-05-02 21:46:46.052072,a465831b-20b0-4d63-a64b-e95ea11e769a,200.000000000,False,pay_g3dsdbce2tyehixszqigsy3pwa,checkoutdotcom_us1,a465831b-20b0-4d63-a64b-e95ea11e769a,e5b3705b-b816-4df2-83a5-5a9d7b08a41f,CARDPROVIDER,visa,US,US,EVIDENCE_REQUIRED,DISPUTE,0,1,2022-05-19 23:55:35.126,2022-05-19
1,ecba3042-0c98-4765-b98d-6b71ca491b9c,2022-05-02 21:46:38.144,2022-05-02 21:46:46.052072,a465831b-20b0-4d63-a64b-e95ea11e769a,200.000000000,False,pay_g3dsdbce2tyehixszqigsy3pwa,checkoutdotcom_us1,a465831b-20b0-4d63-a64b-e95ea11e769a,e5b3705b-b816-4df2-83a5-5a9d7b08a41f,CARDPROVIDER,visa,US,US,10.4,FRAUD,1,0,2022-05-19 23:55:35.131,2022-05-19
2,ecba3042-0c98-4765-b98d-6b71ca491b9c,2022-05-02 21:46:38.144,2022-05-02 21:46:46.052072,a465831b-20b0-4d63-a64b-e95ea11e769a,200.000000000,False,pay_g3dsdbce2tyehixszqigsy3pwa,checkoutdotcom_us1,a465831b-20b0-4d63-a64b-e95ea11e769a,e5b3705b-b816-4df2-83a5-5a9d7b08a41f,CARDPROVIDER,visa,US,US,EVIDENCE_REQUIRED,DISPUTE,0,1,2022-05-19 23:55:35.507,2022-05-19
3,ecba3042-0c98-4765-b98d-6b71ca491b9c,2022-05-02 21:46:38.144,2022-05-02 21:46:46.052072,a465831b-20b0-4d63-a64b-e95ea11e769a,200.000000000,False,pay_g3dsdbce2tyehixszqigsy3pwa,checkoutdotcom_us1,a465831b-20b0-4d63-a64b-e95ea11e769a,e5b3705b-b816-4df2-83a5-5a9d7b08a41f,CARDPROVIDER,visa,US,US,10.4,FRAUD,1,0,2022-05-19 23:55:35.510,2022-05-19
4,2cb4b676-809d-414c-bff7-91446cbbd515,2022-05-09 22:34:58.444,2022-05-09 22:35:19.726327,223d330d-e662-423b-9359-91f0001bb7d6,625.000000000,False,pay_oc5j373fraculcg6xf6vo5jb5i,checkoutdotcom_us1,223d330d-e662-423b-9359-91f0001bb7d6,d4f912b3-52db-4cf6-9ba1-eaf51af94d66,CARDPROVIDER,visa,US,US,EVIDENCE_REQUIRED,DISPUTE,0,1,2022-05-14 23:55:58.585,2022-05-14


## Filter cards by `card_type` and `billing_country`

In [100]:
cards = trnx_pay_all_df[(trnx_pay_all_df.card_type.isin(['visa', 'mastercard']))&
                        (trnx_pay_all_df.billing_country.isin(['US', 'GB']))]

In [101]:
cards.head()

,id,created_at,inserted_at,beneficiary,usd_amount,is_recharge,payment_reference,processing_account,pay_id,user_id,partner,card_type,card_issuer_country,billing_country,event_name,event_type,is_fraud,is_dispute,fraud_reported_date,fraud_reported_date_day
0,ecba3042-0c98-4765-b98d-6b71ca491b9c,2022-05-02 21:46:38.144,2022-05-02 21:46:46.052072,a465831b-20b0-4d63-a64b-e95ea11e769a,200.000000000,False,pay_g3dsdbce2tyehixszqigsy3pwa,checkoutdotcom_us1,a465831b-20b0-4d63-a64b-e95ea11e769a,e5b3705b-b816-4df2-83a5-5a9d7b08a41f,CARDPROVIDER,visa,US,US,EVIDENCE_REQUIRED,DISPUTE,0,1,2022-05-19 23:55:35.126,2022-05-19
1,ecba3042-0c98-4765-b98d-6b71ca491b9c,2022-05-02 21:46:38.144,2022-05-02 21:46:46.052072,a465831b-20b0-4d63-a64b-e95ea11e769a,200.000000000,False,pay_g3dsdbce2tyehixszqigsy3pwa,checkoutdotcom_us1,a465831b-20b0-4d63-a64b-e95ea11e769a,e5b3705b-b816-4df2-83a5-5a9d7b08a41f,CARDPROVIDER,visa,US,US,10.4,FRAUD,1,0,2022-05-19 23:55:35.131,2022-05-19
2,ecba3042-0c98-4765-b98d-6b71ca491b9c,2022-05-02 21:46:38.144,2022-05-02 21:46:46.052072,a465831b-20b0-4d63-a64b-e95ea11e769a,200.000000000,False,pay_g3dsdbce2tyehixszqigsy3pwa,checkoutdotcom_us1,a465831b-20b0-4d63-a64b-e95ea11e769a,e5b3705b-b816-4df2-83a5-5a9d7b08a41f,CARDPROVIDER,visa,US,US,EVIDENCE_REQUIRED,DISPUTE,0,1,2022-05-19 23:55:35.507,2022-05-19
3,ecba3042-0c98-4765-b98d-6b71ca491b9c,2022-05-02 21:46:38.144,2022-05-02 21:46:46.052072,a465831b-20b0-4d63-a64b-e95ea11e769a,200.000000000,False,pay_g3dsdbce2tyehixszqigsy3pwa,checkoutdotcom_us1,a465831b-20b0-4d63-a64b-e95ea11e769a,e5b3705b-b816-4df2-83a5-5a9d7b08a41f,CARDPROVIDER,visa,US,US,10.4,FRAUD,1,0,2022-05-19 23:55:35.510,2022-05-19
4,2cb4b676-809d-414c-bff7-91446cbbd515,2022-05-09 22:34:58.444,2022-05-09 22:35:19.726327,223d330d-e662-423b-9359-91f0001bb7d6,625.000000000,False,pay_oc5j373fraculcg6xf6vo5jb5i,checkoutdotcom_us1,223d330d-e662-423b-9359-91f0001bb7d6,d4f912b3-52db-4cf6-9ba1-eaf51af94d66,CARDPROVIDER,visa,US,US,EVIDENCE_REQUIRED,DISPUTE,0,1,2022-05-14 23:55:58.585,2022-05-14


In [41]:
# cards_march = cards

In [102]:
cards_may = cards

## Concatenate Cards DFs

In [103]:
cards_concat = pd.concat([cards_march, 
                          cards_april, 
                          cards_may])

## Cards statistics

In [104]:
cards_gr = cards_concat.groupby([ 'fraud_reported_date_day', 
                                  'partner',  
                                  'processing_account',
                                  'card_type', 
                                  'billing_country']).agg(fraud_cnt=('is_fraud', 'sum'),
                                                          dispute_cnt=('is_dispute', 'sum'),
                                                          total_cnt=('is_fraud', 'count'), 
                                                          fraud_rario = ('is_fraud', lambda x: x.sum()/x.count()*100.),
                                                          dispute_rario = ('is_dispute', lambda x: x.sum()/x.count()*100.)
                                                         )
                                                                                                                                                                                    

In [105]:
cards_gr

fraud_cnt  \
fraud_reported_date_day partner      processing_account card_type  billing_country              
2022-03-05              CARDPROVIDER checkoutdotcom_us1 visa       US                       2   
2022-03-08              CARDPROVIDER checkoutdotcom_us1 visa       US                      10   
2022-03-10              CARDPROVIDER checkoutdotcom_us1 visa       US                      10   
2022-03-11              CARDPROVIDER checkoutdotcom_us1 visa       US                       4   
2022-03-13              CARDPROVIDER checkoutdotcom_us1 visa       US                      14   
2022-03-14              CARDPROVIDER checkoutdotcom_us1 visa       US                      12   
2022-03-15              CARDPROVIDER checkoutdotcom_us1 visa       US                      18   
2022-03-16              CARDPROVIDER checkoutdotcom_us1 mastercard US                       0   
                                                        visa       US                       6   
2022-03-17              CARDPROVIDER checkoutdotcom_us1 visa       US                       2   
2022-03-23              CARDPROVIDER checkoutdotcom_us1 visa       US                       2   
2022-03-24              CARDPROVIDER checkoutdotcom_us1 visa       US                       2   
2022-03-25              CARDPROVIDER checkoutdotcom_us1 visa       US                       2   
2022-03-26              CARDPROVIDER checkoutdotcom_us1 visa       US                       0   
2022-03-30              CARDPROVIDER checkoutdotcom_us1 mastercard US                       0   
2022-04-07              CARDPROVIDER checkoutdotcom_us1 visa       US                       2   
2022-04-08              CARDPROVIDER checkoutdotcom_us1 visa       US                      14   
2022-04-09              CARDPROVIDER checkoutdotcom_us1 visa       US                       0   
2022-04-15              CARDPROVIDER checkoutdotcom_us1 visa       US                       6   
2022-04-17              CARDPROVIDER checkoutdotcom_us1 visa       US                       2   
2022-04-20              CARDPROVIDER checkoutdotcom_us1 visa       US                       2   
2022-04-21              CARDPROVIDER checkoutdotcom_us1 visa       US                      26   
2022-04-22              CARDPROVIDER checkoutdotcom_us1 mastercard GB                       0   
2022-04-23              CARDPROVIDER checkoutdotcom_us1 mastercard GB                       0   
2022-04-26              CARDPROVIDER checkoutdotcom_us1 mastercard GB                       0   
                                                        visa       US                       2   
2022-04-27              CARDPROVIDER checkoutdotcom_us1 mastercard GB                       0   
                                                        visa       US                       4   
2022-04-28              CARDPROVIDER checkoutdotcom_us1 visa       US                       0   
2022-04-29              CARDPROVIDER checkoutdotcom_us1 mastercard GB                       0   
                                                        visa       US                       0   
2022-04-30              CARDPROVIDER checkoutdotcom_us1 visa       US                       2   
2022-05-13              CARDPROVIDER checkoutdotcom_us1 visa       US                       4   
2022-05-14              CARDPROVIDER checkoutdotcom_us1 visa       US                       0   
2022-05-19              CARDPROVIDER checkoutdotcom_us1 visa       US                       6   
2022-05-20              CARDPROVIDER checkoutdotcom_us1 visa       GB                       0   
2022-05-21              CARDPROVIDER checkoutdotcom_us1 visa       US                       6   

                                                                                    dispute_cnt  \
fraud_reported_date_day partner      processing_account card_type  billing_country                
2022-03-05              CARDPROVIDER checkoutdotcom_us1 visa       US                         2   
202

In [65]:
cards_gr.to_excel("./csv/cards_processing_account_fraud_ratio.xlsx")